# ARA coupling-graph test on Gemma 4 (Colab)

Runs the framework's coupling-structure test on **Gemma 4** using Colab's free GPU.
The weights download to **Google's** machine, not yours.

**Before running:** open https://huggingface.co/google/gemma-4-E2B , accept the license, and make a **Read** access token (Settings -> Access Tokens).

Steps: (1) install, (2) log in, (3) run the test, (4) read the metrics against the blind prediction (LLM_GEMMA4_BLIND_PREDICTION.md).

Set Runtime -> Change runtime type -> **T4 GPU** first.

In [ ]:
%pip install -q -U "transformers>=4.57" torch accelerate huggingface_hub

In [ ]:
from huggingface_hub import login
login()  # paste your READ token (accept the Gemma 4 license on the model card first)

In [ ]:
# ---- the coupling-graph test (same prompt/seed/steps as the Pythia size-series) ----
import time, numpy as np, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

PROMPT = "The framework proposes that natural oscillating systems"
N_STEPS, SEED = 200, 42
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

def coupling_test(model_id):
    print('loading', model_id)
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=DTYPE, attn_implementation='eager',
        device_map='auto' if torch.cuda.is_available() else None,
        output_hidden_states=True, output_attentions=True).eval()
    cfg = model.config; tc = getattr(cfg, 'text_config', cfg)
    nL = getattr(cfg,'num_hidden_layers',None) or tc.num_hidden_layers
    nH = getattr(cfg,'num_attention_heads',None) or tc.num_attention_heads
    hid = getattr(cfg,'hidden_size',None) or tc.hidden_size
    pm = round(sum(p.numel() for p in model.parameters())/1e6)
    print(f'  {nL} layers, {nH} heads, {hid} dim, ~{pm}M params')
    NODES = [('ln',L,None) for L in range(nL+1)] + [('head',L,H) for L in range(nL) for H in range(nH)]
    n = len(NODES)
    torch.manual_seed(SEED)
    dev = next(model.parameters()).device
    cur = tok(PROMPT, return_tensors='pt').input_ids.to(dev)
    ts = np.zeros((n, N_STEPS), np.float32); past=None; t0=time.time()
    with torch.no_grad():
        for s in range(N_STEPS):
            o = model(cur, past_key_values=past, use_cache=True,
                      output_hidden_states=True, output_attentions=True)
            past = o.past_key_values; idx=0
            for L in range(nL+1):
                ts[idx,s]=float(torch.linalg.norm(o.hidden_states[L][0,-1].float())); idx+=1
            for L in range(nL):
                A = o.attentions[L] if (o.attentions is not None and o.attentions[L] is not None) else None
                for H in range(nH):
                    ts[idx,s]=float(A[0,H,-1,:].max()) if A is not None else 0.0; idx+=1
            lg=o.logits[0,-1].float(); tv,ti=torch.topk(lg,40); tv=tv-tv.max()
            p=torch.softmax(tv,-1)
            nxt = ti[0:1] if (torch.isnan(p).any() or (p<0).any()) else ti[torch.multinomial(p,1)]
            cur = nxt.unsqueeze(0)
    el=time.time()-t0; print(f'  gen {el:.1f}s')
    sd=ts.std(1); al=sd>1e-6; na=int(al.sum())
    z=(ts-ts.mean(1,keepdims=True))/(sd[:,None]+1e-9); z[~al]=0
    C=np.clip(np.nan_to_num((z@z.T)/N_STEPS),-1,1); np.fill_diagonal(C,1)
    nl=np.array([L if L is not None else -1 for _,L,_ in NODES])
    wi,ac=[],[]
    for i in range(n):
        if not al[i]: continue
        s_=(nl==nl[i])&(np.arange(n)!=i)&al; d_=(nl!=nl[i])&al
        if s_.any(): wi.append(C[i,s_].mean())
        if d_.any(): ac.append(C[i,d_].mean())
    ev=sorted(np.linalg.eigvalsh(C),reverse=True)
    adj=(np.abs(C)>0.85)&~np.eye(n,dtype=bool); Ai=adj.astype(np.int32)
    tri=int(np.trace(Ai@Ai@Ai)//6); deg=adj.sum(1); u2=int(((deg<2)&al).sum())
    xl=int(sum(1 for i in range(n) for j in range(i+1,n) if al[i] and al[j] and C[i,j]>0.85 and nl[i]!=nl[j]))
    closure=tri/max(na,1); loose=u2/max(na,1)
    r=dict(model=model_id, n_layers=int(nL), params_M=pm, alive_pct=round(100*na/n,1),
           within_across=round(float(np.mean(wi)/max(np.mean(ac),1e-9)),3) if wi and ac else 0,
           spectral_decay=round(float(ev[1]/max(ev[0],1e-9)),3), n_anti=int((C<-0.5).sum()//2),
           cross_layer_pos=xl, closure_ratio=round(closure,3), loose_fraction=round(loose,3),
           intel_index=round(closure/max(loose,1e-3),3))
    print('  ->', r); return r

# smallest first; uncomment 12B (48 layers) once E2B works. Repo ids are CASE-SENSITIVE.
RESULTS = []
for mid in ['google/gemma-4-E2B']:
    try: RESULTS.append(coupling_test(mid))
    except Exception as e: print('FAILED', mid, e)
# for mid in ['google/gemma-4-12b']:  RESULTS.append(coupling_test(mid))   # accept its license too

In [ ]:
# Compare to the Pythia baseline + score the blind predictions
import pandas as pd
print('Pythia baseline (LLM_SIZE_SERIES_RESULT.md): Pythia-410M / 24 layers was the champion at within/across=1.51, spectral_decay=0.42, 3520 anti-phase.')
df = pd.DataFrame(RESULTS)
display(df)
for r in RESULTS:
    print(f"\n{r['model']} ({r['n_layers']} layers):")
    print(f"  P1 depth-not-width: within/across={r['within_across']} (beat 1.51? {r['within_across']>1.51})")
    print(f"  P3 shared-KV -> cross-layer: cross_layer_pos={r['cross_layer_pos']}, closure={r['closure_ratio']}")
    print(f"  P4 dual-RoPE -> anti-phase: n_anti={r['n_anti']}")
    print(f"  P5 per-layer-emb -> engaged: alive={r['alive_pct']}%, loose={r['loose_fraction']}")
print('\nPaste these numbers back into the chat to fill the results table + score P1-P5.')